
E-Commerce 고객 행동 기반 구매 전환 분석 (다시 생각해야할듯)
===

In [24]:
import pandas as pd
import numpy as np
from datetime import datetime

import seaborn as sns
import matplotlib.pyplot as plt
# 모두다 나오게 출력

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

Load Tables
---

In [53]:

# 파일 로드
customer = pd.read_csv('data/customer.csv')
product = pd.read_csv('data/product.csv', on_bad_lines='skip')
transactions = pd.read_csv('data/transactions.csv')
click_stream = pd.read_csv('data/click_stream.csv')


Explore Table
---

In [47]:
# Customer 테이블 기본 구조 확인
customer.info()
print(customer.shape)
print(customer.dtypes)
customer.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   customer_id         100000 non-null  int64  
 1   first_name          100000 non-null  str    
 2   last_name           100000 non-null  str    
 3   username            100000 non-null  str    
 4   email               100000 non-null  str    
 5   gender              100000 non-null  str    
 6   birthdate           100000 non-null  str    
 7   device_type         100000 non-null  str    
 8   device_id           100000 non-null  str    
 9   device_version      100000 non-null  str    
 10  home_location_lat   100000 non-null  float64
 11  home_location_long  100000 non-null  float64
 12  home_location       100000 non-null  str    
 13  home_country        100000 non-null  str    
 14  first_join_date     100000 non-null  str    
dtypes: float64(2), int64(1), str(12)
memory usage:

,customer_id,first_name,last_name,username,email,gender,birthdate,device_type,device_id,device_version,home_location_lat,home_location_long,home_location,home_country,first_join_date
0,2870,Lala,Maryati,671a0865-ac4e-4dc4-9c4f-c286a1176f7e,671a0865_ac4e_4dc4_9c4f_c286a1176f7e@startupca...,F,1996-06-14,iOS,c9c0de76-0a6c-4ac2-843f-65264ab9fe63,iPhone; CPU iPhone OS 14_2_1 like Mac OS X,-1.043345,101.360523,Sumatera Barat,Indonesia,2019-07-21
1,8193,Maimunah,Laksmiwati,83be2ba7-8133-48a4-bbcb-b46a2762473f,83be2ba7_8133_48a4_bbcb_b46a2762473f@zakyfound...,F,1993-08-16,Android,fb331c3d-f42e-40fe-afe2-b4b73a8a6e25,Android 2.2.1,-6.212489,106.818850,Jakarta Raya,Indonesia,2017-07-16
2,7279,Bakiman,Simanjuntak,3250e5a3-1d23-4675-a647-3281879d42be,3250e5a3_1d23_4675_a647_3281879d42be@startupca...,M,1989-01-23,iOS,d13dde0a-6ae1-43c3-83a7-11bbb922730b,iPad; CPU iPad OS 4_2_1 like Mac OS X,-8.631607,116.428436,Nusa Tenggara Barat,Indonesia,2020-08-23


In [38]:
# product 테이블 기본 구조 확인
product.info()
print(product.shape)
print(product.dtypes)
product.head(3)


<class 'pandas.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  44424 non-null  int64  
 1   gender              44424 non-null  str    
 2   masterCategory      44424 non-null  str    
 3   subCategory         44424 non-null  str    
 4   articleType         44424 non-null  str    
 5   baseColour          44409 non-null  str    
 6   season              44403 non-null  str    
 7   year                44423 non-null  float64
 8   usage               44107 non-null  str    
 9   productDisplayName  44417 non-null  str    
dtypes: float64(1), int64(1), str(8)
memory usage: 3.4 MB
(44424, 10)
id                      int64
gender                    str
masterCategory            str
subCategory               str
articleType               str
baseColour                str
season                    str
year                  float64
usag

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch


In [39]:
# transactions 테이블 기본 구조 확인
transactions.info()
print(transactions.shape)
print(transactions.dtypes)
transactions.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 852584 entries, 0 to 852583
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   created_at              852584 non-null  str    
 1   customer_id             852584 non-null  int64  
 2   booking_id              852584 non-null  str    
 3   session_id              852584 non-null  str    
 4   product_metadata        852584 non-null  str    
 5   payment_method          852584 non-null  str    
 6   payment_status          852584 non-null  str    
 7   promo_amount            852584 non-null  int64  
 8   promo_code              326536 non-null  str    
 9   shipment_fee            852584 non-null  int64  
 10  shipment_date_limit     852584 non-null  str    
 11  shipment_location_lat   852584 non-null  float64
 12  shipment_location_long  852584 non-null  float64
 13  total_amount            852584 non-null  int64  
dtypes: float64(2), int64(4), str(8)

,created_at,customer_id,booking_id,session_id,product_metadata,payment_method,payment_status,promo_amount,promo_code,shipment_fee,shipment_date_limit,shipment_location_lat,shipment_location_long,total_amount
0,2018-07-29T15:22:01.458193Z,5868,186e2bee-0637-4710-8981-50c2d737bc42,3abaa6ce-e320-4e51-9469-d9f3fa328e86,"[{'product_id': 54728, 'quantity': 1, 'item_pr...",Debit Card,Success,1415,WEEKENDSERU,10000,2018-08-03T05:07:24.812676Z,-8.227893,111.969107,199832
1,2018-07-30T12:40:22.365620Z,4774,caadb57b-e808-4f94-9e96-8a7d4c9898db,2ee5ead1-f13e-4759-92df-7ff48475e970,"[{'product_id': 16193, 'quantity': 1, 'item_pr...",Credit Card,Success,0,NaN,10000,2018-08-03T01:29:03.415705Z,3.013470,107.802514,155526
2,2018-09-15T11:51:17.365620Z,4774,6000fffb-9c1a-4f4a-9296-bc8f6b622b50,93325fb6-eb00-4268-bb0e-6471795a0ad0,"[{'product_id': 53686, 'quantity': 4, 'item_pr...",OVO,Success,0,NaN,10000,2018-09-18T08:41:49.422380Z,-2.579428,115.743885,550696


In [82]:
# click_stream 테이블 기본 구조 확인
click_stream.info(show_counts=True)
print(click_stream.shape)
print(click_stream.dtypes)
click_stream.head(3)


<class 'pandas.DataFrame'>
RangeIndex: 12833602 entries, 0 to 12833601
Data columns (total 6 columns):
 #   Column          Non-Null Count     Dtype              
---  ------          --------------     -----              
 0   session_id      12833602 non-null  str                
 1   event_name      12833602 non-null  str                
 2   event_time      12833602 non-null  datetime64[us, UTC]
 3   event_id        12833602 non-null  str                
 4   traffic_source  12833602 non-null  str                
 5   event_metadata  4289540 non-null   str                
dtypes: datetime64[us, UTC](1), str(5)
memory usage: 587.5 MB
(12833602, 6)
session_id                        str
event_name                        str
event_time        datetime64[us, UTC]
event_id                          str
traffic_source                    str
event_metadata                    str
dtype: object


,session_id,event_name,event_time,event_id,traffic_source,event_metadata
0,fb0abf9e-fd1a-44dd-b5c0-2834d5a4b81c,HOMEPAGE,2019-09-06 15:54:32.821085+00:00,9c4388c4-c95b-4678-b5ca-e9cbc0734109,MOBILE,NaN
1,fb0abf9e-fd1a-44dd-b5c0-2834d5a4b81c,SCROLL,2019-09-06 16:03:57.821085+00:00,4690e1f5-3f99-42d3-84a5-22c4c4d8500a,MOBILE,NaN
2,7d440441-e67a-4d36-b324-80ffd636d166,HOMEPAGE,2019-09-01 12:05:10.322763+00:00,88aeaeb5-ec98-4859-852c-8abb483faf31,MOBILE,NaN


Data Preprocessing 
---

+ #### 날짜 타입 변경

In [54]:
# 날짜 Type 변경
customer['birthdate'] = pd.to_datetime(customer['birthdate'], errors='coerce')
transactions['created_at'] = pd.to_datetime(transactions['created_at'], errors='coerce')
transactions['shipment_date_limit'] = pd.to_datetime(transactions['shipment_date_limit'], errors='coerce')
click_stream['event_time'] = pd.to_datetime(click_stream['event_time'], errors='coerce')

In [57]:
customer.info()
transactions.info()
click_stream.info()
transactions.head()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   customer_id         100000 non-null  int64         
 1   first_name          100000 non-null  str           
 2   last_name           100000 non-null  str           
 3   username            100000 non-null  str           
 4   email               100000 non-null  str           
 5   gender              100000 non-null  str           
 6   birthdate           100000 non-null  datetime64[us]
 7   device_type         100000 non-null  str           
 8   device_id           100000 non-null  str           
 9   device_version      100000 non-null  str           
 10  home_location_lat   100000 non-null  float64       
 11  home_location_long  100000 non-null  float64       
 12  home_location       100000 non-null  str           
 13  home_country        100000 non-null  str 

,created_at,customer_id,booking_id,session_id,product_metadata,payment_method,payment_status,promo_amount,promo_code,shipment_fee,shipment_date_limit,shipment_location_lat,shipment_location_long,total_amount
0,2018-07-29 15:22:01.458193+00:00,5868,186e2bee-0637-4710-8981-50c2d737bc42,3abaa6ce-e320-4e51-9469-d9f3fa328e86,"[{'product_id': 54728, 'quantity': 1, 'item_pr...",Debit Card,Success,1415,WEEKENDSERU,10000,2018-08-03 05:07:24.812676+00:00,-8.227893,111.969107,199832
1,2018-07-30 12:40:22.365620+00:00,4774,caadb57b-e808-4f94-9e96-8a7d4c9898db,2ee5ead1-f13e-4759-92df-7ff48475e970,"[{'product_id': 16193, 'quantity': 1, 'item_pr...",Credit Card,Success,0,NaN,10000,2018-08-03 01:29:03.415705+00:00,3.013470,107.802514,155526
2,2018-09-15 11:51:17.365620+00:00,4774,6000fffb-9c1a-4f4a-9296-bc8f6b622b50,93325fb6-eb00-4268-bb0e-6471795a0ad0,"[{'product_id': 53686, 'quantity': 4, 'item_pr...",OVO,Success,0,NaN,10000,2018-09-18 08:41:49.422380+00:00,-2.579428,115.743885,550696
3,2018-11-01 11:23:48.365620+00:00,4774,f5e530a7-4350-4cd1-a3bc-525b5037bcab,bcad5a61-1b67-448d-8ff4-781d67bc56e4,"[{'product_id': 20228, 'quantity': 1, 'item_pr...",Credit Card,Success,0,NaN,0,2018-11-05 17:42:27.954235+00:00,-3.602334,120.363824,271012
4,2018-12-18 11:20:30.365620+00:00,4774,0efc0594-dbbf-4f9a-b0b0-a488cfddf8a2,df1042ab-13e6-4072-b9d2-64a81974c51a,"[{'product_id': 55220, 'quantity': 1, 'item_pr...",Credit Card,Success,0,NaN,0,2018-12-23 17:24:07.361785+00:00,-3.602334,120.363824,198753


+ #### 최근 동향을 파악하기 위해 4개년치 추출

In [74]:
# 4개년 계산
max_year = pd.to_datetime(transactions['created_at']).max().year  #2021
years = [max_year - i for i in range(4)]
start_year = min(years)

# 각 연도별 필터링
filtered_transactions = transactions[transactions['created_at'].dt.year >= start_year]
filtered_click_stream = click_stream[click_stream['event_time'].dt.year >= start_year]


In [78]:
# 결과 요약 출력
print("기준 연도:", years)
print("거래 수:", len(filtered_transactions))
print("클릭 로그 수:", len(filtered_click_stream))
print(filtered_transactions['created_at'].min())
print(filtered_click_stream ['event_time'].min())

기준 연도: [2022, 2021, 2020, 2019]
거래 수: 740513
클릭 로그 수: 11126385
2019-01-01 00:04:56.716253+00:00
2019-01-01 00:00:01.231240+00:00


+ #### 결측치 처리

In [79]:
#결측치 처리
filtered_transactions['promo_amount'] = filtered_transactions['promo_amount'].fillna(0)
filtered_transactions['promo_code'] = filtered_transactions['promo_code'].fillna('None')
filtered_click_stream['traffic_source'] = filtered_click_stream['traffic_source'].fillna('Unknown')

In [83]:
filtered_transactions.info()
filtered_click_stream.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 740513 entries, 5 to 852583
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   created_at              740513 non-null  datetime64[us, UTC]
 1   customer_id             740513 non-null  int64              
 2   booking_id              740513 non-null  str                
 3   session_id              740513 non-null  str                
 4   product_metadata        740513 non-null  str                
 5   payment_method          740513 non-null  str                
 6   payment_status          740513 non-null  str                
 7   promo_amount            740513 non-null  int64              
 8   promo_code              740513 non-null  str                
 9   shipment_fee            740513 non-null  int64              
 10  shipment_date_limit     740513 non-null  datetime64[us, UTC]
 11  shipment_location_lat   740513 non-null  f

+ #### 이상치처리
    결제 주체 신뢰성 확보를 위해 **만 10세 미만 데이터는 입력 오류로 판단**  
    분석 모수에서 제외

In [92]:
#이상치 처리
#나이 이상치 확인
customer['age'] = 2021 - customer['birthdate'].dt.year  #거래 기준 max날짜 기준
#display(customer)
print(customer['age'].max())
print(customer['age'].min())    #5 ->10
customer.loc[(customer['age'] < 10)] = np.nan
customer['age'].count()


67.0
10.0


np.int64(99794)

+ #### 필요컬럼 추가 ####
    1. Customer
        - 연령대그룹 : age_groups
    2. Transactions
        - 프로모션 사용 여부 :  promo_used
        - 거래 성공 여부 : is_success
        - 실제거래 금액 : total_non_promo

In [93]:
# Customer
bins = [0, 20, 30, 40, 50, 60, 200]
labels = ['10대', '20대', '30대', '40대', '50대', '60대+']
customer['age_group'] = pd.cut(
    customer['age'], bins=bins, labels=labels, right=False
)

In [97]:
# Transactions
filtered_transactions['promo_used'] = (filtered_transactions['promo_code'].str.lower() != 'none') | (filtered_transactions['promo_amount'] > 0)
filtered_transactions['is_success'] = (filtered_transactions['payment_status'] == 'Success').astype(int)
filtered_transactions['total_non_promo'] = filtered_transactions['total_amount'] - filtered_transactions['promo_amount']

In [98]:
#customer.head()
filtered_transactions.head()

,created_at,customer_id,booking_id,session_id,product_metadata,payment_method,payment_status,promo_amount,promo_code,shipment_fee,shipment_date_limit,shipment_location_lat,shipment_location_long,total_amount,promo_used,is_success,total_non_promo
5,2019-02-03 11:25:55.365620+00:00,4774,1ed58c46-67fb-4386-924b-983c74ccb4d7,7fa0b583-6d30-40bc-8b61-0f70f5bef30f,"[{'product_id': 59620, 'quantity': 1, 'item_pr...",Debit Card,Success,6369,WEEKENDMANTAP,5000,2019-02-07 10:41:59.997463+00:00,-3.602334,120.363824,181865,True,1,175496
6,2019-03-22 11:53:02.365620+00:00,4774,c0ced313-e6b1-4a2a-b21f-347eccda5f96,e140f1f1-6da1-42b3-b2d1-56ac6fc72d4a,"[{'product_id': 53136, 'quantity': 1, 'item_pr...",Credit Card,Success,0,None,10000,2019-03-27 14:40:13.181562+00:00,-7.712608,110.502877,306599,False,1,306599
7,2019-05-08 11:29:21.365620+00:00,4774,0ed6730f-a5db-4e40-9a54-c343474d872c,e7eff973-b499-4b8d-9892-89e67e5fd7d4,"[{'product_id': 14142, 'quantity': 1, 'item_pr...",Credit Card,Success,0,None,10000,2019-05-10 20:09:58.104955+00:00,3.293114,98.857043,153913,False,1,153913
8,2019-06-24 12:08:52.365620+00:00,4774,6edd9366-4788-4bbb-ae5b-744e45e6118d,0d20c278-e082-4f28-9afe-8b5907d7a284,"[{'product_id': 12282, 'quantity': 1, 'item_pr...",Credit Card,Success,4019,AZ2022,10000,2019-06-28 10:34:36.588498+00:00,-6.126016,106.779552,370757,True,1,366738
9,2019-08-10 12:14:24.365620+00:00,4774,771665e9-7df2-41c8-8168-16d5cdd407a1,f5352cdc-0e3c-4e77-af42-645501dcded4,"[{'product_id': 53051, 'quantity': 1, 'item_pr...",Credit Card,Success,9072,WEEKENDSERU,10000,2019-08-14 18:07:04.125993+00:00,-0.992004,114.438739,1295740,True,1,1286668


In [ ]:
# 연-월 컬럼 추가 (예: 2023-11)
filtered_transactions['year_month'] = filtered_transactions['created_at'].dt.to_period('M')

# 연월별 거래량과 매출 합계 집계
monthly_stats = (filtered_transactions.groupby('year_month').agg(
    transaction_count=('created_at', 'count'),  # 거래 건수
    sales_sum=('total_amount', 'sum')                 # 매출 합계
    ).reset_index()
)

# 결과 확인
display(monthly_stats)

C:\Users\최주리\AppData\Local\Temp\ipykernel_38392\1448108817.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  filtered_transactions['year_month'] = filtered_transactions['created_at'].dt.to_period('M')


,year_month,transaction_count,sales_sum
0,2019-11,11347,6168207639
1,2019-12,11209,5969762660
2,2020-11,16788,9313364180
3,2020-12,16545,9294827011
4,2021-11,25063,13879522102
5,2021-12,24873,13664906373


In [ ]:
filtered_transactions.to_csv("output/transactions.csv", index=False, encoding='utf-8-sig')
filtered_click_stream.to_csv("output/click_stream.csv", index=False, encoding='utf-8-sig')
customer.to_csv("output/customer.csv", index=False, encoding='utf-8-sig')
product.to_csv("output/product.csv", index=False, encoding='utf-8-sig')